In [1]:
from dotenv import load_dotenv
from groq import Groq
import os
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [2]:
from ingest import load_faq_data, build_index
documents = load_faq_data()
index = build_index(documents)

In [3]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages = messages )
response.choices[0].message.content

'Absolutely! I’d be happy to help you get started. To make sure I give you the most accurate information, could you let me know which specific course you’re interested in? \n\nIf you already have a course name or code, just share that, and I can walk you through:\n\n1. **Eligibility & Prerequisites** – Any required background or prior courses.\n2. **Enrollment Steps** – How to register, any deadlines, and where to sign up.\n3. **Course Details** – Duration, format (self‑paced, live sessions, hybrid), and any required materials.\n4. **Support & Resources** – Where to find help if you run into any issues.\n\nOnce I have a bit more detail, I’ll give you a step‑by‑step guide to join the course right away. Looking forward to hearing from you!'

Define search tool

In [4]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Next we tell the model about this function. The model doesn't see our Python code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

In [5]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

The description is the most important field, because the model reads it to decide when to call the function. parameters is a JSON schema for the arguments, and we mark query as required so the model always fills it in.

Sending the question with the tool

In [6]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages,
    tools=[search_tool],
)

response

ChatCompletion(id='chatcmpl-c4a22245-8a54-4003-b750-0ffb52529c02', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='The user asks: "I just discovered the course. Can I join it?" We need to answer according to the course FAQ. Likely answer: Yes, you can join as a participant; registration details: open to anyone, just sign up, etc. Provide information about enrollment, prerequisites, cost, etc. The policy: provide info based on FAQ. Might need to search FAQ. Let\'s search.', tool_calls=[ChatCompletionMessageToolCall(id='fc_94a6042e-f830-4435-b126-957b94e3a3cd', function=Function(arguments='{"query":"join course enrollment"}', name='search'), type='function')]))], created=1782044413, model='openai/gpt-oss-120b', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_803c0ba83d', usage=CompletionUsag

To check if tool was called

In [7]:
# Check if tool was called
if response.choices[0].message.tool_calls:
    print("Tool was called!")
    # Access the tool call details
    tool_call = response.choices[0].message.tool_calls[0]
    print(f"Tool name: {tool_call.function.name}")
    print(f"Arguments: {tool_call.function.arguments}")
else:
    print("No tool was used.")

Tool was called!
Tool name: search
Arguments: {"query":"join course enrollment"}


The LLM is a text model - it can suggest calling a function but can't actually run Python code

It can only describe what function to call and with what arguments

Executing the function and sending the result back

In [8]:
import json

# Extract the tool call
tool_call = response.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)

# Execute the search function
results = search(**args)
result_json = json.dumps(results, indent=2)

In [9]:
# Append the assistant's tool call to messages
messages.append({
    "role": "assistant",
    "content": None,
    "tool_calls": response.choices[0].message.tool_calls
})

# Append the tool result
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result_json
})

In [10]:
# Get final response
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages,
    tools=[search_tool],
)
response.choices[0].message.content

'Yes—you can still join the LLM\u202fZoomcamp. The course is open‑ended, so you can start learning right away and submit homework while the enrollment form is still open.  \n\nIf you want a certificate, you’ll need to finish the “live” cohort (i.e., submit your capstone project before the submission deadline and complete the required peer‑reviews). Otherwise, you can simply work through the materials at your own pace.'

Flow:

User: "I just discovered the course. Can I join it?"

Model: "I should search the FAQ" → Returns tool call

Your code: Executes search("join course enrollment") → Gets FAQ entries

Your code: Sends results back to model

Model: Reads FAQ entries and gives accurate answer about enrollment process